In [ ]:
# Install required packages
!pip install python-jose cryptography langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# A3 – Query Enhancement (Rewriting + HyDE)

- **Adapted from:** `all_rag_techniques/query_transformations.ipynb`, `all_rag_techniques/HyDe_Hypothetical_Document_Embedding.ipynb`
- **Experiment ID:** `A3_QUERY`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Compare original query, single rewrite, and HyDE as retrieval queries. Chunking uses best from A1/A2. Retriever remains dense-only, same Top-K.

## Hypothesis

- Rewrite improves Hit Rate and MRR by adding domain terms.
- HyDE helps semantic queries but may hurt exact-identifier queries.
- Original query works best for queries already containing specific terms.

## Control Variables

- Chunking: best from A1/A2
- Retriever: dense only
- Top-K: same
- Prompt: same naive prompt
- LLM: same

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A3_QUERY"
NOTEBOOK = "04_query_enhancement.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]
TOP_K = config["baseline"]["top_k"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f"LLM OK: {llm.invoke('hi').content[:30]}")
print(f"Embedding OK: dim={len(embeddings.embed_query('test'))}")

## 2. Load Corpus, Build Index, Load Questions

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Load docs
raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

# Chunk (use best from A1/A2 - UPDATE this value)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)
for c in chunks:
    c.page_content = c.page_content.replace('\t', ' ')

# Build single index (shared across all query strategies)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"Index: {vectorstore.index.ntotal} vectors from {len(chunks)} chunks")

# Load questions
with open(PROJECT_ROOT / config["paths"]["questions"], "r", encoding="utf-8") as f:
    eval_questions = json.load(f)
print(f"Questions: {len(eval_questions)}")

## 3. Define Query Strategies

In [ ]:
# Strategy 1: Original query (baseline)
def strategy_original(question: str) -> dict:
    """Use original query directly."""
    return {"retrieval_query": question, "llm_calls": 0, "transform_time": 0.0}


# Strategy 2: Query Rewrite
REWRITE_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""Rewrite the following question to be more specific and suitable for searching a knowledge base about climate change.
Keep all key entities, identifiers, and constraints. Return only the rewritten query.

Original: {question}
Rewritten:"""
)
rewrite_chain = REWRITE_PROMPT | llm


def strategy_rewrite(question: str) -> dict:
    """Rewrite query for better retrieval."""
    with Timer() as t:
        result = rewrite_chain.invoke({"question": question})
    return {
        "retrieval_query": result.content.strip(),
        "llm_calls": 1,
        "transform_time": t.elapsed,
    }


# Strategy 3: HyDE
HYDE_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""Write a short paragraph (3-5 sentences) that would be a perfect answer to this question, as if it appeared in a textbook about climate change.

Question: {question}
Hypothetical answer:"""
)
hyde_chain = HYDE_PROMPT | llm


def strategy_hyde(question: str) -> dict:
    """Generate hypothetical document and use it as retrieval query."""
    with Timer() as t:
        result = hyde_chain.invoke({"question": question})
    return {
        "retrieval_query": result.content.strip(),
        "llm_calls": 1,
        "transform_time": t.elapsed,
    }


STRATEGIES = {
    "ORIGINAL": strategy_original,
    "REWRITE": strategy_rewrite,
    "HYDE": strategy_hyde,
}

print(f"Strategies to test: {list(STRATEGIES.keys())}")

## 4. Run Evaluation

In [ ]:
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)
answer_chain = NAIVE_PROMPT | llm

all_results = []

for strategy_name, strategy_fn in STRATEGIES.items():
    print(f"\n{'='*60}")
    print(f"STRATEGY: {strategy_name}")
    print(f"{'='*60}")

    for q in eval_questions:
        qid = q["question_id"]
        question = q["question"]
        relevant_docs = q.get("relevant_documents", [])

        # Transform query
        transform_result = strategy_fn(question)
        retrieval_query = transform_result["retrieval_query"]

        # Retrieve using transformed query
        with Timer() as t_ret:
            docs = retriever.invoke(retrieval_query)

        retrieved_ids = [d.metadata.get("source", f"chunk_{i}") for i, d in enumerate(docs)]
        context = "\n\n".join([d.page_content for d in docs])

        # Generate answer using ORIGINAL question (not the transformed one)
        with Timer() as t_gen:
            response = answer_chain.invoke({"context": context, "question": question})

        metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=TOP_K)

        record = build_result_record(
            experiment_id=f"{EXPERIMENT_ID}_{strategy_name}",
            notebook=NOTEBOOK,
            config_hash=CONFIG_HASH,
            seed=SEED,
            question_id=qid,
            question=question,
            answer=response.content,
            latency={
                "query_transform_seconds": transform_result["transform_time"],
                "retrieval_seconds": t_ret.elapsed,
                "generation_seconds": t_gen.elapsed,
                "total_seconds": transform_result["transform_time"] + t_ret.elapsed + t_gen.elapsed,
            },
            usage={
                "context_chars": len(context),
                "llm_calls": transform_result["llm_calls"] + 1,
            },
            metrics=metrics,
            query_strategy=strategy_name,
            original_query=question,
            transformed_query=retrieval_query,
        )
        all_results.append(record)

    print(f"  Completed {len(eval_questions)} questions.")

print(f"\nTotal records: {len(all_results)}")

## 5. Comparison Table

In [ ]:
summary_rows = []

for strategy_name in STRATEGIES:
    strat_results = [r for r in all_results if r["query_strategy"] == strategy_name]
    metrics_agg = {}
    for key in strat_results[0]["metrics"]:
        vals = [r["metrics"][key] for r in strat_results if r["metrics"].get(key) is not None]
        metrics_agg[key] = round(np.mean(vals), 3) if vals else None

    avg_transform = np.mean([r["latency"]["query_transform_seconds"] for r in strat_results])
    avg_total = np.mean([r["latency"]["total_seconds"] for r in strat_results])
    avg_llm_calls = np.mean([r["usage"]["llm_calls"] for r in strat_results])

    summary_rows.append({
        "strategy": strategy_name,
        "avg_transform_s": round(avg_transform, 3),
        "avg_total_s": round(avg_total, 3),
        "avg_llm_calls": round(avg_llm_calls, 1),
        **metrics_agg,
    })

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

## 6. Query Drift Check

Verify that rewrite/HyDE did not change the intent.

In [ ]:
print("\nSample query transformations:")
print("-" * 60)
for r in all_results:
    if r["query_strategy"] != "ORIGINAL" and r["question_id"] == "Q001":
        print(f"[{r['query_strategy']}]")
        print(f"  Original : {r['original_query']}")
        print(f"  Transform: {r['transformed_query'][:150]}")
        print()

## 7. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]
save_jsonl(all_results, output_dir / "A3_query_enhancement.jsonl")
save_csv_summary(summary_rows, output_dir / "A3_query_enhancement_summary.csv")
save_config_snapshot(config, output_dir)

## 8. Observations

- Original query: baseline performance, no extra latency.
- Rewrite: `___` improvement in Hit Rate, `___`s extra latency per query.
- HyDE: `___` effect on semantic queries, `___` effect on identifier queries.
- HyDE content is NOT evidence — it should never appear in citations.
- Best strategy for A4+: `___`

_Fill in after running._